In [ ]:
### IMPORTS
import pandas as pd
import numpy as np
from backtester.pnl import pnl
from data_utils.io import load_partitioned_parquet
from data_utils.paths import make_data_path
from backtester.metrics.performance_report import PerformanceReport

In [2]:
###CONFIG PLEASE
INITAL_CAPITAL = 100000
LEVERAGE = 1.0
DELAY_BARS = 1 # 1-bar execution delay
MAKER_FEE = 0.000200
TAKER_FEE = 0.000550
SYMBOL = 'BTCUSDT'
INTERVAL = 60
START = '01/01/2026'
END = None

In [3]:
### THis needs to be a function too
### LOAD DATA

cols = ['mark_close'] # Whatever I want for the strategy 
path_ohlcv = make_data_path('ohlcv', SYMBOL, INTERVAL)
df_ohlcv = load_partitioned_parquet(path_ohlcv, start=START, end=END, columns=cols)

if df_ohlcv.empty:
    raise ValueError("df_ohlcv loaded empty. Check path or date filters.")

path_funding = make_data_path('funding', SYMBOL)
df_funding = load_partitioned_parquet(path_funding,start=START, end=END)

if df_funding.empty:
    raise ValueError("df_funding loaded empty. Check path or date filters.")

df_merge = df_ohlcv.merge(df_funding, how='left', on='timestamp')
df_merge['fundingRate'] = df_merge['fundingRate'].fillna(0)
df_merge = df_merge.sort_index()

mark_close = df_merge["mark_close"].astype("float64").to_numpy()
funding  = df_merge["fundingRate"].astype("float64").to_numpy()


In [4]:
pos = np.ones(len(mark_close)) # always long lol
print(len(pos))

1851


In [5]:
pos = np.zeros_like(mark_close)
for i in range(len(mark_close)):
    if i % 2 == 0:
        pos[i] = 1

In [6]:
pos = np.random.uniform(-1, 1, size=(len(mark_close)))

In [7]:
pnl_df = pnl(df_merge, pos, capital=INITAL_CAPITAL,leverage=LEVERAGE, delay_bars=DELAY_BARS)
print(pnl_df.head(10))
print(pnl_df.tail(5))

                           asset_change (%)  signal (% of equity)  \
timestamp                                                           
2026-01-01 00:00:00+00:00          0.000000              0.619338   
2026-01-01 01:00:00+00:00          0.001777              0.771389   
2026-01-01 02:00:00+00:00         -0.000459              0.975026   
2026-01-01 03:00:00+00:00         -0.000638             -0.897391   
2026-01-01 04:00:00+00:00         -0.003008              0.595110   
2026-01-01 05:00:00+00:00          0.001283              0.170119   
2026-01-01 06:00:00+00:00         -0.000590              0.280598   
2026-01-01 07:00:00+00:00         -0.000341              0.190721   
2026-01-01 08:00:00+00:00          0.001633              0.680384   
2026-01-01 09:00:00+00:00          0.001227              0.104377   

                           held_pos (% of equity)  trade (% of equity)  \
timestamp                                                                
2026-01-01 00:00:00+00:

In [ ]:
report=PerformanceReport(pnl_df)
print(report.position.avg_holding_period)

